# 04. 대화형 Agent 완성

---
## 0. 설치 & 경로

In [ ]:
!uv add -U duckduckgo-search
#!uv add numpy

Resolved 235 packages in 1ms
Checked 210 packages in 53ms


In [5]:
!uv add openai python-dotenv langchain langchain-openai langchain-core langchain-community langchain-text-splitters langchain-classic faiss-cpu pypdf pytz yfinance requests ddgs tabulate

Resolved 235 packages in 3ms
Checked 210 packages in 20ms


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

RAG_SAMPLES = os.path.join(os.getcwd(), 'samples', 'rag_samples')
print('RAG_SAMPLES:', RAG_SAMPLES)
for p in sorted(Path(RAG_SAMPLES).glob('*.pdf')):
    print(' -', p.name)

RAG_SAMPLES: d:\autornd\SK Autonomous R&D\실습\5일차\samples\rag_samples
 - news.pdf
 - 학칙.pdf


---
## Step 1. RAG VectorStore 

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import os

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 학칙.pdf
loader = PyPDFLoader(os.path.join(RAG_SAMPLES, '학칙.pdf'))
all_splits = text_splitter.split_documents(loader.load())

# news.pdf
loader_news = PyPDFLoader(os.path.join(RAG_SAMPLES, 'news.pdf'))
news_splits = text_splitter.split_documents(loader_news.load())
all_splits.extend(news_splits)

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large', api_key=OPENAI_API_KEY
)

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

C:\Users\Admin\AppData\Local\Temp\ipykernel_7204\1377301399.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading existing Chroma store


---
## Step 2. Retriever → Tool (5.3 마지막 섹션)

`langchain_core.tools.create_retriever_tool` — 5.3과 동일합니다.

In [3]:
from langchain_core.tools import create_retriever_tool

search_pdf_tool = create_retriever_tool(
    retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)

---
## Step 3. 유틸 Tool (5.2 복습)

시각 · 주가 — `@tool` 로 등록합니다.

In [4]:
from datetime import datetime
import pytz
import yfinance as yf
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    return f'{timezone} ({location}) 현재시각 {now}'

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""
    history = yf.Ticker(ticker).history(period=period)
    return history.tail(3).to_string() if not history.empty else '데이터 없음'

print(get_current_time.invoke({'timezone': 'Asia/Seoul', 'location': '서울'}))

Asia/Seoul (서울) 현재시각 2026-06-26 17:22:38


---
## Step 4. DuckDuckGo 웹 검색

최신 뉴스·시사는 PDF RAG가 아니라 **웹 검색** Tool을 씁니다.  
Agent가 질문마다 `search_pdf_documents` vs `duckduckgo_results_json` 중 선택합니다.

In [5]:
from langchain_community.tools import DuckDuckGoSearchResults

web_search = DuckDuckGoSearchResults(num_results=3)
print(web_search.name, '-', web_search.description[:80], '...')

duckduckgo_results_json - A wrapper around Duck Duck Go Search. Useful for when you need to answer questio ...


In [6]:
web_search.invoke("26년 월드컵 한국 vs 멕시코 점수는?")

'snippet: MEX 멕시코 01:00 KOR 대한민국 1차 예선· A조· 과달라하라 스타디움(과달라하라) USA 미국 19:00 AUS 호주 1차 예선· D조· 시애틀 스타디움(시애틀) SCO ..., title: 경기 | FIFA 월드컵 26, link: https://www.fifa.com/ko/tournaments/mens/worldcup/canadamexicousa2026/scores-fixtures, snippet: May 19, 2026 · 대한민국과 멕시코의 역대 A매치 전적은 15전 4승 3무 8패 로, 멕시코가 크게 앞서 있습니다. 가장 최근 맞대결은 2025년 9월 미국 내슈빌에서 열린 평가전으로, 2대 2 무승부를 기록했습니다. 특히 두 팀은 월드컵 본선에서 두 차례 만나 모두 한국이 패했습니다., title: 2026 북중미 월드컵 A조 대한민국 vs 멕시코 경기일정·역대전적, link: https://postnpost.com/worldcup-2026-korea-mexico/, snippet: Jun 15, 2026 · 한국 멕시코전이 중요한 이유 이번 경기는 사실상 A조 1위 결정전 으로 평가받고 있습니다. 현재 한국과 멕시코는 나란히 승점 3점을 기록하며 조 선두권을 형성하고 있습니다., title: 2026 월드컵 한국 멕시코전 일정·중계 총정리｜16강 분수령 되는 이유..., link: https://cha4se.tistory.com/entry/2026-월드컵-한국-멕시코전-일정·중계-총정리｜16강-분수령-되는-이유-예상-라인업'

---
## Step 5. AgentExecutor — Tool 통합

5.2 수동 tool 루프 대신 **AgentExecutor**가 tool_calls 루프를 자동 처리합니다.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json']


In [8]:
# PDF → search_pdf_documents
r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사학위과정 수업연한'}`


34
[별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드
문과대학
국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German Language and LiteraturePANTONE 5315C불어불문학과문학석사M.A. in French Language and LiteraturePANTONE 5315C노어노문학과문학석사M.A. in Russian Language and LiteraturePANTONE 5315C사학과문학석사M.A. in HistoryPANTONE 5315C철학과문학석사M.A. in PhilosophyPANTONE 5315C문헌정보학과문헌정보학석사M.A. in Library and Information SciencePANTONE 5315C기록관리학석사M.A. in Record Management and ArchivesPANTONE 5315C심리학과문학석사M.A. in PsychologyPANTONE 5315C심리학석사M.S. in PsychologyPANTONE 5315C인지과학석사M.S. in Cognitive SciencePANTONE 5315C
협동과정

협동과정
비교문학협동과정문학석사M.A. in Comparative LiteraturePANTONE 5315C인지과학협동과정인지과학석사M.S. in Cognitive SciencePANTONE 5315C문학석사M.A. in Cognit

In [9]:
# 시각 → get_current_time
r = executor.invoke({'input': '부산 지금 몇시야?', 'chat_history': []})
print('시각:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_time` with `{'timezone': 'Asia/Seoul', 'location': '부산'}`


Asia/Seoul (부산) 현재시각 2026-06-26 17:22:49부산의 현재 시각은 2026년 6월 26일 17시 22분 49초입니다.

> Finished chain.
시각: 부산의 현재 시각은 2026년 6월 26일 17시 22분 49초입니다.


---
## Step 6. 멀티턴 — `chat_history` (5.1 + Agent)

In [10]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('학칙에서 석사 수업연한 알려줘'))
print('2:', session.ask('방금 답한 내용 한 줄로 다시'))



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사 수업연한'}`


Ⅰ. 대학원 학칙  /  15

Ⅰ. 대학원 학칙  /  27

Ⅰ. 대학원 학칙  /  23석사 수업연한에 대한 정보는 학칙 문서에서 확인할 수 있습니다. 구체적인 내용은 해당 문서를 참고하시기 바랍니다. 추가적인 질문이 있으시면 말씀해 주세요!

> Finished chain.
1: 석사 수업연한에 대한 정보는 학칙 문서에서 확인할 수 있습니다. 구체적인 내용은 해당 문서를 참고하시기 바랍니다. 추가적인 질문이 있으시면 말씀해 주세요!


> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사 수업연한'}`


Ⅰ. 대학원 학칙  /  15

Ⅰ. 대학원 학칙  /  27

Ⅰ. 대학원 학칙  /  23석사 수업연한에 대한 정보는 대학원 학칙 문서에서 확인할 수 있습니다.

> Finished chain.
2: 석사 수업연한에 대한 정보는 대학원 학칙 문서에서 확인할 수 있습니다.


---
## Step 7. `input()` 대화 루프

> 주석 해제 후 실행하거나 `python chatbot_agent.py`

In [ ]:
print('종료: exit')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('AI:', session.ask(user_input))

종료: exit


---
## Step 8. 최종 Python 파일

```bash
cd 5일차
python chatbot_agent.py
```